In [1]:
import pandas as pd
import os 
import numpy as np

In [2]:
import sys
sys.path.append(os.path.abspath('..'))
#from meter_data import validate_base_path
import meter_data as md

In [3]:
# define the base path to thumb drive (input path)
basepath = '/Volumes/Untitled/MeterDataTest'
#base_path = '/Desktop/MeterDataTest'

# check if base path exists
if not md.validate_base_path(basepath):
    print(f"Error: Path {basepath} does not exist")
    exit()

In [4]:
# load list of meter data dataframes from CSV files in basepath
meters_df = md.load_meter_dfs(basepath)

In [5]:
# combine list of all meter dataframes into one dataframe
combined_df = md.concat_meter_dfs(meters_df)

# convert dataframe to csv, this will be for processing kw
combined_df.to_csv('meter_data_raw.csv', index=False)

In [6]:
# processesing kwh (interpolation)
processed_kwh = md.process_kwh(combined_df)

# convert (processed kwh/interpolated meter data) dataframe to csv
processed_kwh.to_csv('processed_kwh.csv', index=False)

In [10]:
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')
df
df['datetime'] = pd.to_datetime(df['datetime'])
compair_df = pd.merge(df, processed_kwh, how='outer')

In [11]:
print(compair_df)

                   datetime                meter_name         kwh  is_exact  \
0       2025-07-23 09:40:50              admin_serv_1   1381508.0     False   
1       2025-07-23 09:40:50         admin_serv_2_main    394674.0     False   
2       2025-07-23 09:40:55         ag_science_main_1   1137847.0     False   
3       2025-07-23 09:40:55         ag_science_main_2   8778611.0     False   
4       2025-07-23 09:40:55            ag_science_mcc  12562263.0     False   
...                     ...                       ...         ...       ...   
8779463 2025-10-17 11:39:01        spalding_hall_main  10562008.0     False   
8779464 2025-10-17 11:39:01       student_health_main    140348.0     False   
8779465 2025-10-17 11:39:01  transportation_srvc_main    201282.0     False   
8779466 2025-10-17 11:39:02   univ_high_school_3_main    723953.0     False   
8779467 2025-10-17 11:39:02         wist_annex_1_main    657702.0     False   

         interpolated  
0               False  
1  

In [ ]:
# iterate through the subfolders in the MeterDataTest folder
for subfolder in os.listdir(base_path):
    # create path for each subfolder
    folder_path = os.path.join(base_path, subfolder)

    if not os.path.isdir(folder_path):
        continue
    
    # get the name of the meter from the subfolder name, make lowercase
    meter_name = subfolder.lower().replace(" ", "_").replace("_mtr", "")
    #print(meter_name)

    # list of csv file paths in subfolder
    # addition with the 'and not' is to make sure to ignore the hidden ._ files
    csv_paths = [os.path.join(folder_path, f) 
                 for f in os.listdir(folder_path) 
                 if f.endswith('.csv')
                 and not f.startswith("._")
                 and not f.startswith(".")]

    # empty meter's dataframe list
    meter_dfs = []
    
    # convert each csv to a df, fix columns and add df to df list
    for csv in csv_paths:
        temp_df = pd.read_csv(csv, encoding="utf-8")

        temp_df.columns = temp_df.columns.str.strip().str.lower().str.replace(" ", "_")
        
        # rename columns if they exist
        # some meters have have the different label but they are synonymous
        if '3_phase_positive_real_energy_used' in temp_df.columns:
            temp_df.rename(columns={
                '3_phase_positive_real_energy_used': 'total_watt_hour',
                '3_phase_real_power':'3_phase_watt_total'
            }, inplace=True)

        # error in scripts, total_watt_hour is actually total kwh
        temp_df.rename(columns={'total_watt_hour': 'kwh'}, inplace=True)
        
        # reorder the columns
        temp_df = temp_df[['datetime', 'kwh', '3_phase_watt_total']]
        temp_df.insert(1, 'meter_name', meter_name)

        meter_dfs.append(temp_df)

    # combine all csvs for this meter into one dataframe
    df = pd.concat(meter_dfs, ignore_index=True)
    
    # save the dataframe with all columns to list
    save_data.append(df.copy())

    # PROCESSING KWH DATA (interpolation):

    # drop the 3_phase_watt_total column as its not needed for kwh interpolation
    df.drop('3_phase_watt_total', axis=1, inplace=True)

    # convert datetime column to a datetime type
    df['datetime'] = pd.to_datetime(df['datetime'], format='%Y-%m-%d %H:%M:%S')

    # create column that contains the closest interval for each timestamp (contains ymd hms, using timedelta)
    #df['interval_15min'] = pd.to_datetime(df['datetime'].dt.round('15min'), format='%Y-%m-%d %H:%M:%S')
    df['interval_15min'] = df['datetime'].dt.round('15min')
    
    # create column that contains the offset in seconds from the closest interval for each timestamp
    # - is if its before it and + is if its after
    df['interval_offset'] = (df['datetime'] - df['interval_15min']).dt.total_seconds()

    # create new column with true if an exact interval and false if not
    df['is_exact'] = df['datetime'].eq(df['interval_15min'])
    
    df['interpolated'] = False
    
    interpolated_rows = []

    # interval = the 15min bucket val, group = all rows in that bucket
    for interval, group in df.groupby('interval_15min'):
        # only select rows in group with is_exact == True
        exact = group[group['is_exact']]

        if exact.empty:
            before = group[group['interval_offset'] <= 0]
            after = group[group['interval_offset'] >= 0]

            # check if there are empties
            if not before.empty and not after.empty:
                # grab the closest data to the interval
                time_before = before.iloc[-1]
                time_after = after.iloc[0]
                
                # calculate the estimated kwh
                # get the slope to 4 decimal places
                reading_diff = time_after['kwh'] - time_before['kwh']

                if reading_diff == 0:
                    estimated_kwh = time_before['kwh']
                else:
                    time_diff = (time_after['datetime'] - time_before['datetime']).total_seconds()
                    slope = round(reading_diff / time_diff, 4)
                    sec_before_interval = (interval - time_before['datetime']).total_seconds()
                    estimated_kwh = time_before['kwh'] + (slope * sec_before_interval)

                # create interpolated row
                new_row = time_before.copy()
                new_row['datetime'] = interval
                new_row['kwh'] = estimated_kwh
                new_row['interval_offset'] = 0
                new_row['is_exact'] = True
                new_row['interpolated'] = True
                
                # add new interpolated row to list
                interpolated_rows.append(new_row)

    # combine interpolated data with dataframe
    if interpolated_rows:
        
        df = pd.concat([df, pd.DataFrame(interpolated_rows)], ignore_index=True)

    df = df.drop(columns=['interval_15min', 'interval_offset'])

    # resort the data to be in order of datetime
    df = df.sort_values(by='datetime').reset_index(drop=True)
    
    all_data.append(df)

In [ ]:
# dataframe for all data
combined_data = pd.DataFrame()
save_data_col = pd.DataFrame()

In [ ]:
# create with all columns to save for later
save_data_col = pd.concat(save_data, ignore_index=True)

In [ ]:
# combine all dataframes in list to one dataframe
combined_data = pd.concat(all_data, ignore_index=True)

In [ ]:
# check there is only one instance
print(combined_data[(combined_data['meter_name'] == 'admin_serv_1_mtr') &
    (combined_data['datetime'] == pd.to_datetime('2025-07-23 09:40:50'))])

In [ ]:
# check for if there exists duplicates
duplicate_data = combined_data[combined_data.duplicated(keep=False)]
print(duplicate_data)

In [ ]:
print(combined_data.head(15))

In [ ]:
# convert dataframe with all collumns to csv to save for later
save_data_col.to_csv('save_data_col.csv', index=False)

In [ ]:
# convert dataframe to csv
combined_data.to_csv('interpolated_meter_data.csv', index=False)

In [ ]:
# list of meters and first timestamp in dataset
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
print(df['meter_name'].unique())
print(df.head(1))

In [ ]:
# sample of interpolated data for admin_serv_1 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding="utf-8")
df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) & (df['meter_name'] == 'admin_serv_1_mtr')]
sample_data.to_csv('sample_data.csv', index=False)

In [ ]:
# sample of interpolated data for biomedical_science_main_a_mtr 2025-09-10
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])
date = pd.to_datetime('2025-09-10').date()
sample_data = df[(df['datetime'].dt.date == date) 
    & (df['meter_name'] == 'biomedical_science_main_a_mtr') 
    & ((df['interpolated'] == True) | ((df['datetime'].dt.minute % 15 == 0) & (df['datetime'].dt.second == 0)))
    ].drop_duplicates(subset=['datetime', 'meter_name'])

In [ ]:
sample_data.to_csv('sample_data.csv', index=False)

In [ ]:
# get just interpolated meter data from all meters on days sept 07-09 2025
df = pd.read_csv('interpolated_meter_data.csv', encoding='utf-8')
df['datetime'] = pd.to_datetime(df['datetime'])

In [ ]:
df = df[df['datetime'].dt.date.isin([
    pd.to_datetime('2025-09-07').date(),
    pd.to_datetime('2025-09-08').date(),
    pd.to_datetime('2025-09-09').date()])]
df = df[df['is_exact'] & True]

In [ ]:
print(df)

In [ ]:
df.to_csv('sept07-09_kwh.csv', index=False)